# Part 4 — GenAI: turn call-centre notes into structured reasons, then recommend


In [ ]:
%pip install -q google-genai pydantic

In [4]:
import json
import time

import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

DATA_DIR = "data/"
MODEL_NAME = "gemini-3.5-flash"
ALLOWED = ["forgot_or_no_reminder", "distance_or_transport", "cost_or_fee",
           "illness_or_emergency", "work_or_schedule", "other"]
API_CALLS = 0   # counts every request sent to Gemini in Tasks 4.1-4.3 (retries included)
print(f"Setup done | model: {MODEL_NAME} | API_CALLS reset to {API_CALLS}")

Setup done | model: gemini-3.5-flash | API_CALLS reset to 0


In [2]:
API_KEY = input("Paste your key: ")
client = genai.Client(api_key=API_KEY, http_options=types.HttpOptions(timeout=60000))   # 60 s per request, never hang
print("Gemini client ready (key not shown).")

Gemini client ready (key not shown).


In [3]:
# Flash models available to this key (a listing request, not a generate call; not counted in API_CALLS)
for m in client.models.list():
    if "flash" in m.name:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash


## Task 4.1 — One validated call


In [5]:
SCHEMA = {
    "type": "object",
    "properties": {"root_cause": {"type": "string"}, "confidence": {"type": "number"},
                   "summary": {"type": "string"}, "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"]}

SYSTEM_TEXT = ("You analyse short call-centre notes written after a patient missed a clinic appointment. "
               "You reply with JSON only, following the given schema.")


def craft_prompt(note_text):
    """CRAFT prompt: Context, Role, Action, Tone, Format."""
    return f"""Context: CareLoop Clinics runs eight outpatient clinics in India. After a patient misses an appointment,
the call centre phones them and writes a short note. Notes may be in English or Hinglish (Hindi written in Latin script).

Role: You are a patient-experience analyst who reads these notes and records why the patient missed the visit.

Action: Read the note below and identify the single main reason for the missed appointment.
Choose root_cause from exactly one of these values:
- forgot_or_no_reminder: the patient forgot, or did not receive a reminder
- distance_or_transport: the clinic is too far, or there was no transport
- cost_or_fee: the fee or cost was the reason
- illness_or_emergency: the patient was ill or had an emergency
- work_or_schedule: work, shifts or other schedule clashes
- other: anything else, including when the note does not say why the patient missed
If the note does not clearly state a reason, answer other rather than guess.

Tone: Factual and neutral. Do not invent details that are not in the note.

Format: JSON with four fields:
- root_cause: one of the allowed values above, written exactly as shown
- confidence: a number from 0 to 1 for how sure you are
- summary: at most 25 words, in English
- recommended_action: one short, practical next step for the clinic

<note>{note_text}</note>
"""


RETRY_WAITS = [20, 40]   # seconds to wait before the 2nd and 3rd attempt


def get_raw_reply(prompt):
    """Send one prompt to Gemini (JSON mode, fixed schema, temperature 0) and return the reply text unchanged.
    thinking_config was removed: this model rejects thinking_level="minimal" (the brief allows removing it).
    A temporary failure (503 overloaded, 500, 429 rate limit, timeout) is retried up to 2 more times;
    every attempt is counted in API_CALLS."""
    global API_CALLS
    for attempt in range(len(RETRY_WAITS) + 1):
        API_CALLS += 1
        print(f"   [call #{API_CALLS}] sending to {MODEL_NAME} (attempt {attempt + 1}) ...", flush=True)
        started = time.time()
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_TEXT, temperature=0,
                    max_output_tokens=1000, response_mime_type="application/json", response_schema=SCHEMA))
            print(f"   [call #{API_CALLS}] reply received in {time.time() - started:.1f} s", flush=True)
            return response.text
        except Exception as e:
            code = getattr(e, "code", None)
            temporary = code in (429, 500, 503) or "timeout" in type(e).__name__.lower()
            print(f"   [call #{API_CALLS}] failed after {time.time() - started:.1f} s: "
                  f"{type(e).__name__} {code or ''} {str(e)[:120]}", flush=True)
            if not temporary or attempt == len(RETRY_WAITS):
                raise
            print(f"   temporary error -> waiting {RETRY_WAITS[attempt]} s, then retrying", flush=True)
            time.sleep(RETRY_WAITS[attempt])


def analyse_note(note_text):
    """Analyse one note with the CRAFT prompt and return the parsed dictionary."""
    return json.loads(get_raw_reply(craft_prompt(note_text)))

In [6]:
notes = pd.read_csv(DATA_DIR + "followup_notes.csv")
n001 = notes.loc[notes["note_id"] == "N001", "note_text"].iloc[0]
print("N001:", n001)

print("Task 4.1: analysing N001 ...", flush=True)
result_n001 = analyse_note(n001)
print(result_n001)
print("Fields:", sorted(result_n001.keys()))
print("API calls so far:", API_CALLS)

N001: Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.
Task 4.1: analysing N001 ...
   [call #1] sending to gemini-3.5-flash (attempt 1) ...
   [call #1] reply received in 54.6 s
{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.95, 'summary': 'The patient missed her appointment because she did not receive a reminder message and forgot. She wants to reschedule for next week.', 'recommended_action': 'Reschedule the patient for next week and investigate the reminder delivery issue.'}
Fields: ['confidence', 'recommended_action', 'root_cause', 'summary']
API calls so far: 1


## Task 4.2 — Validate and process the 10 notes

In [7]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)


def in_allowed_list(insight):
    """Second check, after Pydantic: root_cause must be one of the allowed values."""
    return insight.root_cause in ALLOWED

In [8]:
FIELDS = ["root_cause", "confidence", "summary", "recommended_action"]
records = []
print("Task 4.2: processing the first 10 notes (13 s pause between calls) ...", flush=True)

for n, (_, row) in enumerate(notes.head(10).iterrows(), start=1):
    note_text = row["note_text"]
    print(f"\n[{n}/10] {row['note_id']}", flush=True)
    record = {"note_id": row["note_id"], "appointment_id": row["appointment_id"]}

    # input validation: NaN (blank cell), empty or only spaces -> no API call
    if pd.isna(note_text) or str(note_text).strip() == "":
        record.update(root_cause="other", confidence=0.0, summary="No note recorded",
                      recommended_action="Re-call patient", validation_status="skipped_blank", raw_response="")
        records.append(record)
        print(row["note_id"], "-> skipped_blank (no API call)")
        continue

    raw = ""
    try:
        if len(records) > 0:
            print("   pausing 13 s (free-tier rate limit) ...", flush=True)
            time.sleep(13)                       # free-tier rate limit
        raw = get_raw_reply(craft_prompt(note_text))   # same call as analyse_note, keeping the raw text
        data = json.loads(raw)
        for field in FIELDS:                     # copy whatever the reply contained, no repair
            record[field] = data.get(field)
        insight = FollowupInsight(**data)
        record["validation_status"] = "valid" if in_allowed_list(insight) else "invalid"
    except Exception as e:                       # one bad reply must not stop the loop
        record["validation_status"] = "invalid"
        print(row["note_id"], "-> error:", type(e).__name__, e)
    record["raw_response"] = raw
    records.append(record)
    print(row["note_id"], "->", record.get("root_cause"), "|", record["validation_status"])

analysed = pd.DataFrame(records, columns=["note_id", "appointment_id"] + FIELDS + ["validation_status", "raw_response"])
analysed.to_csv("notes_analysed.csv", index=False)
print("\nSaved notes_analysed.csv:", analysed.shape, "| API calls so far:", API_CALLS)
analysed

Task 4.2: processing the first 10 notes (13 s pause between calls) ...

[1/10] N001
   [call #2] sending to gemini-3.5-flash (attempt 1) ...
   [call #2] reply received in 58.4 s
N001 -> forgot_or_no_reminder | valid

[2/10] N002
   pausing 13 s (free-tier rate limit) ...
   [call #3] sending to gemini-3.5-flash (attempt 1) ...
   [call #3] failed after 0.8 s: ServerError 503 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand 
   temporary error -> waiting 20 s, then retrying
   [call #4] sending to gemini-3.5-flash (attempt 2) ...
   [call #4] reply received in 48.3 s
N002 -> distance_or_transport | valid

[3/10] N003
   pausing 13 s (free-tier rate limit) ...
   [call #5] sending to gemini-3.5-flash (attempt 1) ...
   [call #5] reply received in 54.7 s
N003 -> cost_or_fee | valid

[4/10] N004
N004 -> skipped_blank (no API call)

[5/10] N005
   pausing 13 s (free-tier rate limit) ...
   [call #6] sending to gemini-3.

,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response
0,N001,A0872,forgot_or_no_reminder,0.98,The patient forgot the appointment because she...,Reschedule the appointment for next week and v...,valid,"{""root_cause"":""forgot_or_no_reminder"",""confide..."
1,N002,A0897,distance_or_transport,0.95,Patient missed the appointment due to living f...,Offer tele-consultation options or direct the ...,valid,"{""root_cause"":""distance_or_transport"",""confide..."
2,N003,A0526,cost_or_fee,0.95,Patient skipped the appointment because the co...,Inform the patient about available follow-up d...,valid,"{""root_cause"":""cost_or_fee"",""confidence"":0.95,..."
3,N004,A0926,other,0.00,No note recorded,Re-call patient,skipped_blank,
4,N005,A0534,illness_or_emergency,0.98,Patient missed the appointment due to a high f...,Follow up with the patient in a few days to re...,valid,"{""root_cause"":""illness_or_emergency"",""confiden..."
5,N006,A0507,work_or_schedule,0.98,Patient missed appointment due to an urgent wo...,Reschedule the appointment to an evening slot.,valid,"{""root_cause"":""work_or_schedule"",""confidence"":..."
6,N007,A0333,forgot_or_no_reminder,0.98,The patient forgot the appointment because the...,Reschedule the appointment and verify SMS remi...,valid,"{""root_cause"":""forgot_or_no_reminder"",""confide..."
7,N008,A0222,distance_or_transport,0.98,The patient missed the appointment due to a ci...,Offer a teleconsultation or assist in reschedu...,valid,"{""root_cause"":""distance_or_transport"",""confide..."
8,N009,A0977,other,0.95,The clinic attempted to call the patient three...,Send an SMS or WhatsApp message to the patient...,valid,"{""root_cause"":""other"",""confidence"":0.95,""summa..."
9,N010,A0596,work_or_schedule,0.98,Patient missed the appointment due to a factor...,Reschedule the patient for a Saturday appointm...,valid,"{""root_cause"":""work_or_schedule"",""confidence"":..."


In [9]:
expected = {"N001": "forgot_or_no_reminder", "N002": "distance_or_transport", "N003": "cost_or_fee",
            "N004": "other", "N005": "illness_or_emergency", "N006": "work_or_schedule",
            "N007": "forgot_or_no_reminder", "N008": "distance_or_transport", "N009": "other",
            "N010": "work_or_schedule"}
analysed["expected"] = analysed["note_id"].map(expected)
non_blank = analysed[analysed["validation_status"] != "skipped_blank"]
print("Agreement with the expected root causes (9 non-blank notes):",
      (non_blank["root_cause"] == non_blank["expected"]).sum(), "of", len(non_blank))
print(analysed["validation_status"].value_counts().to_string())

Agreement with the expected root causes (9 non-blank notes): 9 of 9
validation_status
valid            9
skipped_blank    1


## Task 4.3 — Break it on purpose

In [10]:
# (a) two deliberately bad replies
bad_replies = {
    "confidence = 1.7": {"root_cause": "cost_or_fee", "confidence": 1.7,
                         "summary": "Patient found the fee too high.", "recommended_action": "Offer a discount."},
    "missing recommended_action": {"root_cause": "cost_or_fee", "confidence": 0.8,
                                   "summary": "Patient found the fee too high."},
}
for label, reply in bad_replies.items():
    try:
        FollowupInsight(**reply)
        print(label, "-> unexpectedly valid")
    except ValidationError as e:
        print(f"{label} -> caught ValidationError:\n{e}\n")

confidence = 1.7 -> caught ValidationError:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

missing recommended_action -> caught ValidationError:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'cost_or_f...ound the fee too high.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing



In [11]:
# (b) V1 (bare) vs V2 (CRAFT) prompt on N007 (Hinglish) and N009 (ambiguous).
# Same schema, JSON mode and temperature 0 for both; only the prompt differs.
# V2 replies are reused from Task 4.2 (same prompt, same settings) to save free-tier calls.
def v1_prompt(note_text):
    return f"Classify this note: {note_text}"


comparison = []
print("Task 4.3(b): V1 prompt on N007 and N009 (V2 reused from Task 4.2) ...", flush=True)
for note_id in ["N007", "N009"]:
    note_text = notes.loc[notes["note_id"] == note_id, "note_text"].iloc[0]
    print(f"\n{note_id}: pausing 13 s (free-tier rate limit) ...", flush=True)
    time.sleep(13)
    try:
        v1_cause = json.loads(get_raw_reply(v1_prompt(note_text))).get("root_cause")
    except Exception as e:
        v1_cause = f"error: {type(e).__name__}"
    print(f"   V1 root_cause: {v1_cause}", flush=True)
    v2_cause = analysed.loc[analysed["note_id"] == note_id, "root_cause"].iloc[0]
    for version, cause in [("V1 bare", v1_cause), ("V2 CRAFT", v2_cause)]:
        comparison.append({"note": note_id, "prompt": version, "root_cause": cause,
                           "in_allowed_list": cause in ALLOWED})

pd.DataFrame(comparison)

Task 4.3(b): V1 prompt on N007 and N009 (V2 reused from Task 4.2) ...

N007: pausing 13 s (free-tier rate limit) ...
   [call #12] sending to gemini-3.5-flash (attempt 1) ...
   [call #12] reply received in 37.6 s
   V1 root_cause: Lack of SMS reminder and forgetfulness

N009: pausing 13 s (free-tier rate limit) ...
   [call #13] sending to gemini-3.5-flash (attempt 1) ...
   [call #13] failed after 25.6 s: ServerError 503 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand 
   temporary error -> waiting 20 s, then retrying
   [call #14] sending to gemini-3.5-flash (attempt 2) ...
   [call #14] reply received in 47.5 s
   V1 root_cause: Unreachable - phone switched off


,note,prompt,root_cause,in_allowed_list
0,N007,V1 bare,Lack of SMS reminder and forgetfulness,False
1,N007,V2 CRAFT,forgot_or_no_reminder,True
2,N009,V1 bare,Unreachable - phone switched off,False
3,N009,V2 CRAFT,other,True


**Which prompt to deploy:** V2 (CRAFT). The bare V1 prompt never tells the model which categories exist, so any
label it returns is a guess at our vocabulary — and a label outside the allowed list (see `in_allowed_list` above)
cannot be counted. V2 names the allowed values and tells the model to answer `other` instead of guessing, so its
answers can be aggregated across notes, and an ambiguous note like N009 is not forced into a made-up reason.

In [12]:
# (c) total API calls used in Tasks 4.1-4.3
print("Total Gemini API calls:", API_CALLS)

Total Gemini API calls: 14


## Task 4.4 (a) — Merge with the bookings and cross-tabulate

In [13]:
appts = pd.read_csv("appointments_clean.csv")
merged = pd.read_csv("notes_analysed.csv").merge(appts, on="appointment_id", how="left")
print("Merged rows:", len(merged))
pd.crosstab(merged["root_cause"], merged["reminder_sent"], margins=True)

Merged rows: 10


reminder_sent,No,Yes,All
root_cause,,,
cost_or_fee,0,1,1
distance_or_transport,1,1,2
forgot_or_no_reminder,2,0,2
illness_or_emergency,0,1,1
other,1,1,2
work_or_schedule,1,1,2
All,5,5,10
